# Lab 3 - Prompt Engineering

**Name:** Suraj Pujari 
**Roll No:** SCFP125013   
**Programme:** MCA  
**Repository:** `GenAI_LLM_engineering_SCFP125013`

## Objective
In this lab, I will practise prompt engineering through structured summarization, review classification, resume field extraction, customer support writing, translation, and iterative editing.


In [4]:
import importlib.util

packages = {
    "groq": "groq",
    "python-dotenv": "dotenv",
    "ipykernel": "ipykernel"
}

for package_name, module_name in packages.items():
    if importlib.util.find_spec(module_name):
        print(f"✅ {package_name}: Installed")
    else:
        print(f"❌ {package_name}: Not installed")

✅ groq: Installed
✅ python-dotenv: Installed
✅ ipykernel: Installed


In [5]:
%pip install groq

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Setup — Connect to Groq

Before running this notebook:
1. Install the dependencies in the notebook's Python environment: `python -m pip install groq python-dotenv`.
2. Create a `.env` file in this project folder containing `GROQ_API_KEY=your_actual_key_here`.
3. Never paste the real API key into notebook cells or commit `.env` to GitHub.

This cell searches the current folder and its parent folders for `.env`, loads the key without printing it, and creates the Groq client.



> **Execution note:** This notebook contains saved outputs from Suraj's prior execution with the configured environment. Rerun the relevant cells after changing the model, prompts, dependencies, or API configuration. The notebook uses sample/test data for demonstration; the doctor-summary exercise is not for real medical decisions.


In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

current_folder = Path.cwd()
env_file = next(
    (folder / ".env" for folder in (current_folder, *current_folder.parents)
     if (folder / ".env").is_file()),
    None
)

if env_file is None:
    raise FileNotFoundError(
        "Could not find .env. Put it in the project folder and add GROQ_API_KEY=your_key_here."
    )

load_dotenv(env_file, override=False)
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing from the .env file.")

client = Groq(api_key=api_key)
MODEL_NAME = "openai/gpt-oss-20b"

print("Environment file found:", env_file)
print("Groq client is ready.")
print("Model:", MODEL_NAME)

Environment file found: /Users/surajpujari/Data/MIT VUP/Sem 3/GenAI/Activity-3/.env
Groq client is ready.
Model: openai/gpt-oss-20b


## A helper for text generation

All exercises use the same helper. It checks for an empty response rather than silently continuing with blank text.

In [7]:
def generate(prompt, temperature=0.2, max_tokens=700, system_message=None):
    if system_message is None:
        system_message = (
            "Follow the instructions carefully. Use only supplied information for "
            "summaries and extraction. Do not invent missing facts."
        )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_completion_tokens=max_tokens,
        reasoning_effort="low"
    )

    result_text = response.choices[0].message.content
    if not result_text or not result_text.strip():
        raise RuntimeError(
            "Groq returned an empty message. Try increasing max_tokens or check model availability."
        )
    return result_text.strip()

print("Generation helper is ready.")

Generation helper is ready.


### Quick connection test

Run this before starting the exercises. A short sentence confirms that the API is responding.

In [8]:
print(generate(
    "Explain prompt engineering in one short sentence.",
    temperature=0.2,
    max_tokens=300
))

Prompt engineering is the art of crafting input prompts to guide an AI model toward desired outputs.


## Exercise 1  Doctor Summary Generator

**Inputs:** `patient_notes`, `patient_name`  
**Expected output:** The same three sections every time: Symptoms, Diagnosis, and Recommendation.

The prompt uses only the consultation notes. This is an assignment demonstration, not a tool for making real medical decisions.

In [9]:
def generate_doctor_summary(patient_notes, patient_name):
    prompt = f'''
Prepare a clear summary of the consultation notes.

Use exactly this format and keep all three headings in this order:

PATIENT: {patient_name}

SYMPTOMS:
- List the symptoms mentioned in the notes.

DIAGNOSIS:
- State the diagnosis only if it is explicitly mentioned in the notes.
- If no diagnosis is stated, write: Not specified in the notes.

RECOMMENDATION:
- List the recommendations mentioned in the notes.

Use only information in the notes. Do not add a date, diagnosis, symptom, medicine,
or recommendation that is not provided. Keep the wording concise.

Consultation notes:
{patient_notes}
'''
    return generate(
        prompt,
        temperature=0.1,
        max_tokens=700,
        system_message="Format consultation notes for an educational exercise. Do not add medical facts absent from the supplied notes."
    )

notes_1 = (
    "Persistent headache for 2 days, slight fever around 100 F, body aches, fatigue, "
    "trouble sleeping, and no appetite. The notes say it may be a viral infection. "
    "Advice: rest, drink fluids, take ibuprofen for fever, and return if there is no "
    "improvement in 5 days."
)
print(generate_doctor_summary(notes_1, "Rahul Sharma"))

PATIENT: Rahul Sharma

SYMPTOMS:
- Persistent headache for 2 days
- Slight fever around 100 °F
- Body aches
- Fatigue
- Trouble sleeping
- No appetite

DIAGNOSIS:
- Not specified in the notes.

RECOMMENDATION:
- Rest
- Drink fluids
- Take ibuprofen for fever
- Return if there is no improvement in 5 days


In [10]:
notes_2 = (
    "Anita Patel reports sharp right knee pain while climbing stairs. It started two "
    "weeks ago after a fall. There is swelling and slightly reduced range of motion. "
    "X-ray shows no fracture. The notes describe a likely soft tissue injury. "
    "Plan: anti-inflammatory medicine, ice, physiotherapy referral, and follow-up in two weeks."
)
print(generate_doctor_summary(notes_2, "Anita Patel"))

PATIENT: Anita Patel

SYMPTOMS:
- Sharp right knee pain while climbing stairs
- Swelling
- Slightly reduced range of motion

DIAGNOSIS:
- Likely soft tissue injury

RECOMMENDATION:
- Anti-inflammatory medicine
- Ice
- Physiotherapy referral
- Follow-up in two weeks


**Check the result:** Both summaries should include `PATIENT`, `SYMPTOMS`, `DIAGNOSIS`, and `RECOMMENDATION`. Check that the model has not added information absent from the notes.

## Exercise 2  Review Classifier

**Input:** `review_text`  
**Expected output:** One category label.

For this exercise, the labels are:
- `Positive` — mostly praise or satisfaction.
- `Negative` — mostly complaint or dissatisfaction.
- `Mixed` — both positive and negative points, or unclear overall sentiment.

In [11]:
def classify_review(review_text):
    prompt = f'''
Choose exactly one label:
Positive: mostly praise or satisfaction.
Negative: mostly complaint or dissatisfaction.
Mixed: both positive and negative points, or unclear overall sentiment.

Return only Positive, Negative, or Mixed. Do not explain.

Review:
{review_text}
'''
    label = generate(
        prompt, temperature=0, max_tokens=200,
        system_message="Classify the review and return only one allowed label."
    ).strip().strip(" .!\n")

    for allowed in ("Positive", "Negative", "Mixed"):
        if label.lower() == allowed.lower():
            return allowed
    return label

review_1 = "The delivery was quick and the product works exactly as expected."
review_2 = "The item arrived late and customer service did not respond."
review_3 = "The product is useful, but the battery life is disappointing."

print("Review 1:", classify_review(review_1))
print("Review 2:", classify_review(review_2))
print("Review 3:", classify_review(review_3))

Review 1: Positive
Review 2: Negative
Review 3: Mixed


## Exercise 3 - Resume Field Extractor

**Inputs:** `resume_text`, `fields_to_extract`  
**Expected output:** Strict JSON containing only requested fields.

If a field is missing, the model should return `null` rather than guess.

In [12]:
import json

def extract_resume_fields(resume_text, fields_to_extract):
    fields = list(fields_to_extract)
    prompt = f'''
Extract these fields from the resume: {json.dumps(fields)}.

Return one valid JSON object only. Include every requested field as a key and no
other keys. Use null when a value is not present. Do not guess. Do not use
Markdown fences or an explanation.

Resume:
{resume_text}
'''
    raw_result = generate(
        prompt, temperature=0, max_tokens=700,
        system_message="Extract information faithfully and return valid JSON only."
    )
    start, end = raw_result.find("{"), raw_result.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("The model did not return a JSON object. Response: " + raw_result)

    data = json.loads(raw_result[start:end + 1])
    return {field: data.get(field, None) for field in fields}

sample_resume = '''
Suraj Pujari
Email: suraj@example.com
Education: Master of Computer Applications
Skills: Java, Spring Boot, React, MySQL
Project: Payment Gateway built with Spring Boot
'''
requested_fields = ["name", "email", "education", "skills", "phone"]
resume_data = extract_resume_fields(sample_resume, requested_fields)
print(json.dumps(resume_data, indent=2, ensure_ascii=False))

{
  "name": "Suraj Pujari",
  "email": "suraj@example.com",
  "education": "Master of Computer Applications",
  "skills": [
    "Java",
    "Spring Boot",
    "React",
    "MySQL"
  ],
  "phone": null
}


## Exercise 4 - Customer Support Reply Generator

**Inputs:** `customer_message`, `company_name`, `max_words`  
**Expected output:** A polite, consistent reply within the word limit.

In [13]:
def generate_support_reply(customer_message, company_name, max_words):
    if max_words < 1:
        raise ValueError("max_words must be at least 1.")

    prompt = f'''
Write a support reply on behalf of {company_name}.

Customer message:
{customer_message}

Be polite, calm, clear, and helpful. Acknowledge the concern. Keep a professional,
consistent tone. Do not promise an outcome that is not known. Use no more than
{max_words} words. Return only the reply.
'''
    reply = generate(
        prompt, temperature=0.2, max_tokens=500,
        system_message="Write concise, polite customer support replies and obey the word limit."
    )

    if len(reply.split()) > max_words:
        reply = generate(
            f"Rewrite this reply in no more than {max_words} words. Keep it polite. "
            f"Return only the reply:\n\n{reply}",
            temperature=0, max_tokens=500
        )

    words = reply.split()
    if len(words) > max_words:
        reply = " ".join(words[:max_words])
    return reply

support_reply = generate_support_reply(
    "My order has not arrived, even though the tracking page says delivered.",
    "Northstar Store",
    45
)
print(support_reply)
print("Word count:", len(support_reply.split()))

I’m sorry to hear your package hasn’t arrived. Let’s investigate. I’ll check the carrier’s records and contact them for details. I’ll update you within 24 hours with what we find. Thank you for your patience.
Word count: 35


## Exercise 5 - Multi-language Translator

**Inputs:** `sentence`, `target_language`, `formality`  
**Expected output:** A translation that matches the requested formality.

In [14]:
def translate_sentence(sentence, target_language, formality):
    formality = formality.lower().strip()
    if formality not in {"formal", "informal"}:
        raise ValueError("formality must be either 'formal' or 'informal'.")

    prompt = f'''
Translate the sentence into {target_language}.
Formality: {formality}.
Use natural wording for this level of formality and preserve the original meaning.
Do not explain the translation. Return only the translated sentence.

Sentence:
{sentence}
'''
    return generate(
        prompt, temperature=0.1, max_tokens=400,
        system_message="Translate carefully, preserve meaning, and follow the requested formality."
    )

sentence = "Could you please help me with this problem?"
print("Formal:", translate_sentence(sentence, "Hindi", "formal"))
print("Informal:", translate_sentence(sentence, "Hindi", "informal"))

Formal: कृपया क्या आप मेरी इस समस्या में सहायता कर सकते हैं?
Informal: क्या तुम मेरी इस समस्या में मदद करोगे?


## Exercise 6 - Iterative Content Refiner

**Inputs:** `draft_text`, `issue_to_fix`  
**Expected output:** A revised draft that addresses one issue over at least three rounds.

Each round uses the previous version and focuses on the same issue. The notebook keeps every version so the changes can be reviewed.

In [15]:
def refine_draft(draft_text, issue_to_fix, rounds=3):
    if rounds < 3:
        raise ValueError("Run at least 3 rounds for this exercise.")

    versions = [draft_text]
    current_text = draft_text

    for round_number in range(1, rounds + 1):
        prompt = f'''
Revise this draft to fix one issue: {issue_to_fix}
Keep the original meaning and do not introduce unsupported facts.
Make a focused improvement. Return only the revised draft.

Draft:
{current_text}
'''
        current_text = generate(
            prompt, temperature=0.2, max_tokens=700,
            system_message="You are an editor. Make focused revisions and return only the revised text."
        )
        versions.append(current_text)
        print(f"Round {round_number} completed.")

    return versions

draft = (
    "Our app help people to manage their tasks. It have reminders and a simple dashboard. "
    "Users can see what they need to do."
)
refinement_rounds = refine_draft(
    draft_text=draft,
    issue_to_fix="Fix grammar and make the writing clearer.",
    rounds=3
)

for index, version in enumerate(refinement_rounds):
    heading = "Original draft" if index == 0 else f"After round {index}"
    print(f"\n{heading}:\n{version}")

Round 1 completed.
Round 2 completed.
Round 3 completed.

Original draft:
Our app help people to manage their tasks. It have reminders and a simple dashboard. Users can see what they need to do.

After round 1:
Our app helps people manage their tasks. It includes reminders and a simple dashboard, allowing users to see what they need to do.

After round 2:
Our app helps users manage their tasks by providing reminders and a simple dashboard that lets them see what needs to be done.

After round 3:
Our app helps users manage tasks by providing reminders and a simple dashboard that shows what needs to be done.
